In [3]:
import os
import pandas as pd
import requests

# 1. API Endpoint URL
api_url = "https://api.mfapi.in/mf/125497"

headers = {"User-Agent": "Mozilla/5.0"}
params = {}

print("Fetching data from API...")
response = requests.get(api_url, headers=headers, params=params)

# Check if request was successful
if response.status_code == 200:
    # 2. Parse JSON response
    json_data = response.json()

    # The historical NAV array is stored under the 'data' key in this API
    if "data" in json_data:
        df = pd.DataFrame(json_data["data"])
    else:
        df = pd.DataFrame(json_data)

    print(f"Successfully fetched data! Shape: {df.shape}")

    # 3. Define output path in data/raw
    output_dir = "../data/raw" if os.path.exists("../data/raw") else "data/raw"
    os.makedirs(output_dir, exist_ok=True)

    csv_filename = "11_api_raw_data.csv"
    output_path = os.path.join(output_dir, csv_filename)

    # 4. Save DataFrame as CSV
    df.to_csv(output_path, index=False)
    print(f"Saved raw API data to: {output_path}")

    # Display preview
    display(df.head())

else:
    print(
        f"Failed to fetch data. HTTP Status Code: {response.status_code}"
    )
    print(response.text)

Fetching data from API...
✅ Successfully fetched data! Shape: (3129, 2)
💾 Saved raw API data to: ../data/raw\11_api_raw_data.csv


,date,nav
0,24-07-2026,204.85350
1,23-07-2026,205.55390
2,22-07-2026,207.67480
3,21-07-2026,210.09000
4,20-07-2026,209.44660


In [1]:
import os
import pandas as pd
import requests

# 1. Define 5 Key Mutual Fund Schemes (Scheme Code: Scheme Name)
key_schemes = {
    119551: "SBI Bluechip",
    120503: "ICICI Bluechip",
    118632: "Nippon Large Cap",
    119092: "Axis Bluechip",
    120841: "Kotak Bluechip",
}

combined_data = []

# 2. Fetch NAV data for each scheme
for scheme_code, scheme_name in key_schemes.items():
    url = f"https://api.mfapi.in/mf/{scheme_code}"
    print(f"Fetching data for: {scheme_name} ({scheme_code})...")

    res = requests.get(url)
    if res.status_code == 200:
        json_data = res.json()
        meta = json_data.get("meta", {})
        nav_list = json_data.get("data", [])

        # Add scheme metadata to each NAV record
        for item in nav_list:
            item["scheme_code"] = scheme_code
            item["scheme_name"] = meta.get("scheme_name", scheme_name)
            item["fund_house"] = meta.get("fund_house", "")
            item["scheme_category"] = meta.get("scheme_category", "")
            combined_data.append(item)
    else:
        print(f" Failed to fetch code {scheme_code}: HTTP {res.status_code}")

# 3. Create DataFrame
df_5_schemes = pd.DataFrame(combined_data)

# Reorder columns nicely
columns_order = [
    "scheme_code",
    "scheme_name",
    "fund_house",
    "scheme_category",
    "date",
    "nav",
]
df_5_schemes = df_5_schemes[columns_order]

print("\n" + "=" * 60)
print(f" Total Records Fetched across 5 Schemes: {len(df_5_schemes)}")

# 4. Save to data/raw CSV
output_dir = "../data/raw" if os.path.exists("../data/raw") else "data/raw"
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "12_key_5_schemes_nav.csv")
df_5_schemes.to_csv(csv_path, index=False)

print(f" Saved to: {csv_path}")
display(df_5_schemes.head(10))

Fetching data for: SBI Bluechip (119551)...
Fetching data for: ICICI Bluechip (120503)...
Fetching data for: Nippon Large Cap (118632)...
Fetching data for: Axis Bluechip (119092)...
Fetching data for: Kotak Bluechip (120841)...

 Total Records Fetched across 5 Schemes: 16902
 Saved to: ../data/raw\12_key_5_schemes_nav.csv


,scheme_code,scheme_name,fund_house,scheme_category,date,nav
0,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,27-07-2026,106.81260
1,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,24-07-2026,106.59610
2,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,23-07-2026,106.58510
3,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,22-07-2026,106.61070
4,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,21-07-2026,106.65670
5,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,20-07-2026,106.57370
6,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,17-07-2026,106.60010
7,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,16-07-2026,106.58290
8,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,15-07-2026,106.51160
9,119551,Aditya Birla Sun Life Banking & PSU Debt Fund ...,Aditya Birla Sun Life Mutual Fund,Debt Scheme - Banking and PSU Fund,14-07-2026,106.46090


In [2]:
import pandas as pd

# Load fund_master dataset
df_master = pd.read_csv("../data/raw/01_fund_master.csv")

print("==================================================")
print("📌 FUND MASTER EXPLORATION")
print("==================================================\n")

# 1. Unique counts and value lists
print(
    f"1️⃣ UNIQUE FUND HOUSES ({df_master['fund_house'].nunique()} total):"
    if "fund_house" in df_master.columns
    else "1️⃣ FUND HOUSE COLUMN NOT FOUND"
)
if "fund_house" in df_master.columns:
    print(df_master["fund_house"].dropna().unique())

print("\n" + "-" * 50)
print(
    f"2️⃣ UNIQUE CATEGORIES ({df_master['category'].nunique()} total):"
    if "category" in df_master.columns
    else "2️⃣ CATEGORY COLUMN NOT FOUND"
)
if "category" in df_master.columns:
    print(df_master["category"].dropna().unique())

print("\n" + "-" * 50)
print(
    f"3️⃣ UNIQUE SUB-CATEGORIES ({df_master['sub_category'].nunique()} total):"
    if "sub_category" in df_master.columns
    else "3️⃣ SUB_CATEGORY COLUMN NOT FOUND"
)
if "sub_category" in df_master.columns:
    print(df_master["sub_category"].dropna().unique())

print("\n" + "-" * 50)
print(
    f"4️⃣ UNIQUE RISK GRADES ({df_master['risk_grade'].nunique()} total):"
    if "risk_grade" in df_master.columns
    else "4️⃣ RISK_GRADE COLUMN NOT FOUND"
)
if "risk_grade" in df_master.columns:
    print(df_master["risk_grade"].dropna().unique())

print("\n" + "=" * 50)
print("📌 AMFI SCHEME CODE STRUCTURE")
print("==================================================")

code_col = [c for c in df_master.columns if "scheme_code" in c or "code" in c][0]
print(f"Using column: '{code_col}'")
print("Data Type:", df_master[code_col].dtype)
print("Sample Scheme Codes:", df_master[code_col].head(10).tolist())
print(
    "Code length summary:",
    df_master[code_col].astype(str).str.len().value_counts().to_dict(),
)

📌 FUND MASTER EXPLORATION

1️⃣ UNIQUE FUND HOUSES (10 total):
['SBI Mutual Fund' 'HDFC Mutual Fund' 'ICICI Prudential MF'
 'Nippon India MF' 'Kotak Mahindra MF' 'Axis Mutual Fund'
 'Aditya Birla Sun Life MF' 'UTI Mutual Fund' 'Mirae Asset MF'
 'DSP Mutual Fund']

--------------------------------------------------
2️⃣ UNIQUE CATEGORIES (2 total):
['Equity' 'Debt']

--------------------------------------------------
3️⃣ UNIQUE SUB-CATEGORIES (12 total):
['Large Cap' 'Small Cap' 'Gilt' 'Mid Cap' 'Short Duration' 'Value'
 'Liquid' 'Index/ETF' 'Flexi Cap' 'Index' 'Large & Mid Cap' 'ELSS']

--------------------------------------------------
4️⃣ RISK_GRADE COLUMN NOT FOUND

📌 AMFI SCHEME CODE STRUCTURE
Using column: 'amfi_code'
Data Type: int64
Sample Scheme Codes: [119551, 119552, 119598, 119599, 119120, 100016, 125497, 100033, 125498, 100025]
Code length summary: {6: 40}


In [3]:
# Load NAV history dataset
df_nav = pd.read_csv("../data/raw/02_nav_history.csv")

# Identify code columns in both DataFrames
master_code_col = [
    c for c in df_master.columns if "scheme_code" in c or "code" in c
][0]
nav_code_col = [c for c in df_nav.columns if "scheme_code" in c or "code" in c][
    0
]

# Extract unique sets of codes
master_codes = set(df_master[master_code_col].dropna().unique())
nav_codes = set(df_nav[nav_code_col].dropna().unique())

# Validation logic
missing_in_nav = master_codes - nav_codes
match_count = len(master_codes.intersection(nav_codes))
total_master = len(master_codes)
match_percentage = (
    (match_count / total_master) * 100 if total_master > 0 else 0
)

# Output Summary Report
print("==================================================")
print(" DATA QUALITY & VALIDATION SUMMARY")
print("==================================================")
print(f"• Total Unique Schemes in Fund Master: {total_master}")
print(f"• Total Unique Schemes in NAV History: {len(nav_codes)}")
print(f"• Schemes in Master matching NAV History: {match_count}")
print(f"• Scheme Match Rate: {match_percentage:.2f}%")
print(f"• Schemes in Master MISSING from NAV History: {len(missing_in_nav)}")

if missing_in_nav:
    print(f"\n Sample Missing Codes (up to 10): {list(missing_in_nav)[:10]}")
else:
    print("\n PERFECT MATCH: Every code in fund_master exists in nav_history!")
print("==================================================")

📊 DATA QUALITY & VALIDATION SUMMARY
• Total Unique Schemes in Fund Master: 40
• Total Unique Schemes in NAV History: 40
• Schemes in Master matching NAV History: 40
• Scheme Match Rate: 100.00%
• Schemes in Master MISSING from NAV History: 0

✅ PERFECT MATCH: Every code in fund_master exists in nav_history!
